### This notebook analyzes the final 2025 enriched city dataset.
The goal is to compare air-quality patterns across selected global cities. we initially seleteced 18 cities then finalized 15. 
we will ultimately zoom into Dubai to show what city averages hide.

#### Load data + interactive global monthly PM2.5 map + seasonality heatmap

In [17]:
# ============================================================
# Analysis notebook setup
# ============================================================

import sys
import subprocess
import importlib.util
from pathlib import Path

required_packages = [
    "pandas",
    "numpy",
    "plotly",
]

for package in required_packages:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [18]:
# ============================================================
# Project paths
# ============================================================

project_root = Path(".")

data_dir = project_root / "data"
processed_dir = data_dir / "processed"
enriched_dir = processed_dir / "enriched"
combined_enriched_dir = enriched_dir / "combined"
city_selection_dir = processed_dir / "combined" / "city_selection"

figure_output_dir = project_root / "outputs" / "figures"
web_data_output_dir = project_root / "web" / "data"

figure_output_dir.mkdir(parents=True, exist_ok=True)
web_data_output_dir.mkdir(parents=True, exist_ok=True)

In [19]:
# ============================================================
# Load final analysis datasets
# ============================================================

daily_path = combined_enriched_dir / "combined_city_daily_enriched_2025.csv"
monthly_path = combined_enriched_dir / "combined_city_monthly_enriched_2025.csv"
weather_points_path = combined_enriched_dir / "combined_city_weather_points.csv"

selected_main_path = city_selection_dir / "selected_cities_pm25_main.csv"
selected_particle_path = city_selection_dir / "selected_cities_particle_profile.csv"
excluded_context_path = city_selection_dir / "excluded_or_context_cities.csv"

daily = pd.read_csv(daily_path)
monthly = pd.read_csv(monthly_path)
weather_points = pd.read_csv(weather_points_path)

selected_main = pd.read_csv(selected_main_path)
selected_particle = pd.read_csv(selected_particle_path)
excluded_context = pd.read_csv(excluded_context_path)


print("Weather points:", len(weather_points))
print("Selected main cities:", len(selected_main))
print("Particle-profile cities:", len(selected_particle))
print("Excluded/context cities:", len(excluded_context))

Weather points: 15
Selected main cities: 15
Particle-profile cities: 9
Excluded/context cities: 3


In [20]:
# ============================================================
# Basic cleaning for city display
# ============================================================


def clean_city_key(value):
    return str(value).strip().lower().replace(" ", "_").replace("-", "_")


city_display_lookup = {
    "dubai": "Dubai",
    "riyadh": "Riyadh",
    "delhi": "Delhi",
    "lahore": "Lahore",
    "dhaka": "Dhaka",
    "bangkok": "Bangkok",
    "jakarta": "Jakarta",
    "singapore": "Singapore",
    "seoul": "Seoul",
    "tokyo": "Tokyo",
    "beijing": "Beijing",
    "london": "London",
    "los_angeles": "Los Angeles",
    "new_york": "New York",
    "mexico_city": "Mexico City",
    "abu_dhabi": "Abu Dhabi",
    "kuwait_city": "Kuwait City",
    "cape_town": "Cape Town",
}

for df in [
    daily,
    monthly,
    weather_points,
    selected_main,
    selected_particle,
    excluded_context,
]:
    if "city" in df.columns:
        df["city_key"] = df["city"].apply(clean_city_key)
        df["city_display"] = df["city_key"].map(city_display_lookup).fillna(df["city"])

# Ensure date columns
daily["local_date"] = pd.to_datetime(daily["local_date"], errors="coerce")

monthly["month_date"] = pd.to_datetime(
    monthly["local_year"].astype(int).astype(str)
    + "-"
    + monthly["local_month"].astype(int).astype(str).str.zfill(2)
    + "-01"
)

monthly["month_label"] = monthly["month_date"].dt.strftime("%b %Y")

selected_city_keys = selected_main["city_key"].unique()

daily = daily[daily["city_key"].isin(selected_city_keys)].copy()
monthly = monthly[monthly["city_key"].isin(selected_city_keys)].copy()
weather_points = weather_points[
    weather_points["city_key"].isin(selected_city_keys)
].copy()


In [29]:
# ============================================================
# City selection recap table
# ============================================================

selection_recap = selected_main[
    [
        "city_display",
        "country",
        "city_group",
        "first_month",
        "last_month",
        "usable_pm25_months",
        "usable_pm10_months",
        "mean_pm25",
        "mean_pm10",
        "final_city_role",
        "selection_note",
    ]
].copy()

selection_recap = selection_recap.sort_values(
    ["usable_pm25_months", "usable_pm10_months", "city_display"],
    ascending=[False, False, True],
)

selection_recap

,city_display,country,city_group,first_month,last_month,usable_pm25_months,usable_pm10_months,mean_pm25,mean_pm10,final_city_role,selection_note
0,Bangkok,Thailand,High-pollution / fast-growing Asia,2021-04-01,2025-12-01,51,44,22.605649,45.483872,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
1,London,United Kingdom,Policy / developed-city comparison,2021-05-01,2026-01-01,50,50,10.783260,18.060735,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
2,Beijing,China,Dense but policy-relevant Asian comparison,2020-01-01,2026-01-01,37,26,38.929807,62.773763,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
9,Singapore,Singapore,Dense but policy-relevant Asian comparison,2022-10-01,2026-01-01,37,4,15.140140,22.273790,Main PM2.5 comparison,Selected for the main PM2.5 global comparison.
3,Delhi,India,High-pollution / fast-growing Asia,2019-12-01,2025-12-01,36,36,109.556047,214.367794,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
10,Tokyo,Japan,Dense but policy-relevant Asian comparison,2023-07-01,2026-01-01,29,1,9.618451,13.956266,Main PM2.5 comparison,Selected for the main PM2.5 global comparison.
4,Los Angeles,United States,Policy / developed-city comparison,2022-02-01,2025-12-01,28,7,11.015566,23.425332,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
5,Mexico City,Mexico,Optional contrast,2020-10-01,2026-01-01,27,27,20.725139,42.738565,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
11,Lahore,Pakistan,High-pollution / fast-growing Asia,2023-11-01,2025-12-01,25,5,104.672869,304.438779,Main PM2.5 comparison,Selected for the main PM2.5 global comparison.
6,Jakarta,Indonesia,High-pollution / fast-growing Asia,2023-12-01,2025-12-01,24,11,46.173091,56.929578,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...


In [30]:
# Save recap for magazine webpage
selection_recap.to_csv(web_data_output_dir / "selected_city_recap.csv", index=False)

#### Interactive global monthly PM2.5 map

In [32]:
# ============================================================
#  PM2.5 global map
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
import plotly.offline as pyo

# ----------------------------
# Paths
# ----------------------------

project_root = Path(".")

combined_enriched_dir = project_root / "data" / "processed" / "enriched" / "combined"
city_selection_dir = project_root / "data" / "processed" / "combined" / "city_selection"

figure_output_dir = project_root / "outputs" / "figures"
web_data_output_dir = project_root / "web" / "data"

figure_output_dir.mkdir(parents=True, exist_ok=True)
web_data_output_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Load fixed cleaned data
# ----------------------------

monthly_path = combined_enriched_dir / "combined_city_monthly_enriched_2025.csv"
weather_points_path = combined_enriched_dir / "combined_city_weather_points.csv"
selected_main_path = city_selection_dir / "selected_cities_pm25_main.csv"

monthly = pd.read_csv(monthly_path)
weather_points = pd.read_csv(weather_points_path)
selected_main = pd.read_csv(selected_main_path)


# ----------------------------
# Helpers
# ----------------------------


def clean_city_key(value):
    return str(value).strip().lower().replace(" ", "_").replace("-", "_")


city_display_lookup = {
    "dubai": "Dubai",
    "riyadh": "Riyadh",
    "delhi": "Delhi",
    "lahore": "Lahore",
    "dhaka": "Dhaka",
    "bangkok": "Bangkok",
    "jakarta": "Jakarta",
    "singapore": "Singapore",
    "seoul": "Seoul",
    "tokyo": "Tokyo",
    "beijing": "Beijing",
    "london": "London",
    "los_angeles": "Los Angeles",
    "new_york": "New York",
    "mexico_city": "Mexico City",
    "abu_dhabi": "Abu Dhabi",
    "kuwait_city": "Kuwait City",
    "cape_town": "Cape Town",
}


for df in [monthly, weather_points, selected_main]:
    df["city_key"] = df["city"].apply(clean_city_key)
    df["city_display"] = df["city_key"].map(city_display_lookup).fillna(df["city"])

# Fill missing country and city_group for map hover labels

city_to_country = {
    "dubai": "United Arab Emirates",
    "riyadh": "Saudi Arabia",
    "delhi": "India",
    "lahore": "Pakistan",
    "dhaka": "Bangladesh",
    "bangkok": "Thailand",
    "jakarta": "Indonesia",
    "singapore": "Singapore",
    "seoul": "South Korea",
    "tokyo": "Japan",
    "beijing": "China",
    "london": "United Kingdom",
    "los_angeles": "United States",
    "new_york": "United States",
    "mexico_city": "Mexico",
}

city_to_group = {
    "dubai": "Gulf / desert urbanization",
    "riyadh": "Gulf / desert urbanization",
    "delhi": "High-pollution / fast-growing Asia",
    "lahore": "High-pollution / fast-growing Asia",
    "dhaka": "High-pollution / fast-growing Asia",
    "bangkok": "High-pollution / fast-growing Asia",
    "jakarta": "High-pollution / fast-growing Asia",
    "singapore": "Dense but policy-relevant Asian comparison",
    "seoul": "Dense but policy-relevant Asian comparison",
    "tokyo": "Dense but policy-relevant Asian comparison",
    "beijing": "Dense but policy-relevant Asian comparison",
    "london": "Policy / developed-city comparison",
    "los_angeles": "Policy / developed-city comparison",
    "new_york": "Policy / developed-city comparison",
    "mexico_city": "Optional contrast",
}

for df in [monthly, weather_points, selected_main]:
    if "country" not in df.columns:
        df["country"] = pd.NA

    if "city_group" not in df.columns:
        df["city_group"] = pd.NA

    missing_country = df["country"].isna() | (
        df["country"]
        .astype(str)
        .str.lower()
        .str.strip()
        .isin(["", "nan", "none", "null"])
    )

    missing_group = df["city_group"].isna() | (
        df["city_group"]
        .astype(str)
        .str.lower()
        .str.strip()
        .isin(["", "nan", "none", "null"])
    )

    df.loc[missing_country, "country"] = df.loc[missing_country, "city_key"].map(
        city_to_country
    )
    df.loc[missing_group, "city_group"] = df.loc[missing_group, "city_key"].map(
        city_to_group
    )

# Keep only selected main cities
selected_city_keys = selected_main["city_key"].unique()

monthly = monthly[monthly["city_key"].isin(selected_city_keys)].copy()
weather_points = weather_points[
    weather_points["city_key"].isin(selected_city_keys)
].copy()


# Build month columns
monthly["month_date"] = pd.to_datetime(
    monthly["local_year"].astype(int).astype(str)
    + "-"
    + monthly["local_month"].astype(int).astype(str).str.zfill(2)
    + "-01"
)

monthly = monthly.sort_values(["month_date", "city_display"]).copy()
monthly["month_label"] = monthly["month_date"].dt.strftime("%b %Y")

month_order = (
    monthly.drop_duplicates("month_date")
    .sort_values("month_date")["month_label"]
    .tolist()
)


# Remove impossible values just in case
for col in [
    "pm25_monthly_mean",
    "pm25_monthly_median",
    "pm25_monthly_p90",
    "pm10_monthly_mean",
    "pm10_monthly_median",
    "pm10_monthly_p90",
]:
    if col in monthly.columns:
        monthly[col] = pd.to_numeric(monthly[col], errors="coerce")
        monthly.loc[monthly[col] < 0, col] = np.nan


# Main metric for visual comparison
PM25_METRIC = "pm25_monthly_median"
PM25_METRIC_LABEL = "Median PM2.5"


# ============================================================
# 1. Autoplay animated global map using median PM2.5
# ============================================================

map_monthly = monthly.merge(
    weather_points[
        [
            "city_key",
            "weather_latitude",
            "weather_longitude",
        ]
    ].drop_duplicates("city_key"),
    on="city_key",
    how="left",
)

map_monthly = map_monthly[
    map_monthly["weather_latitude"].notna()
    & map_monthly["weather_longitude"].notna()
    & map_monthly[PM25_METRIC].notna()
].copy()

map_monthly["pm25_median_label"] = (
    map_monthly[PM25_METRIC].round(1).astype(str) + " µg/m³"
)

map_monthly["coverage_label"] = (
    map_monthly["pm25_month_coverage_pct"].round(0).astype(str) + "%"
)

if "active_locations_max" in map_monthly.columns:
    map_monthly["marker_size"] = map_monthly["active_locations_max"].fillna(1)
else:
    map_monthly["marker_size"] = 1

map_monthly["marker_size"] = map_monthly["marker_size"].clip(lower=1, upper=8)

# Color range capped for readability, actual values still appear in hover
color_max = np.nanpercentile(map_monthly[PM25_METRIC], 95)
color_max = max(color_max, 40)

fig_map_median = px.scatter_geo(
    map_monthly,
    lat="weather_latitude",
    lon="weather_longitude",
    color=PM25_METRIC,
    size="marker_size",
    animation_frame="month_label",
    category_orders={"month_label": month_order},
    hover_name="city_display",
    hover_data={
        "weather_latitude": False,
        "weather_longitude": False,
        "marker_size": False,
        PM25_METRIC: ":.1f",
        "pm25_month_coverage_pct": ":.0f",
        "valid_pm25_days": True,
        "active_locations_max": True,
        "country": True,
        "city_group": True,
    },
    color_continuous_scale="Inferno_r",
    range_color=[0, color_max],
    projection="natural earth",
    size_max=30,
)

fig_map_median.update_geos(
    showland=True,
    landcolor="rgb(245, 244, 238)",
    showocean=True,
    oceancolor="rgb(228, 238, 247)",
    showcountries=True,
    countrycolor="rgba(90,90,90,0.35)",
    coastlinecolor="rgba(90,90,90,0.35)",
    lataxis_showgrid=True,
    lonaxis_showgrid=True,
)

fig_map_median.update_layout(
    height=720,
    margin=dict(l=20, r=20, t=90, b=20),
    coloraxis_colorbar=dict(title="Median<br>PM2.5<br>µg/m³"),
    title=dict(
        text=(
            "Monthly Median PM2.5 Across Selected Global Cities, 2025"
            "<br><sup>Autoplay map. Marker size reflects active monitoring locations; color shows monthly median PM2.5.</sup>"
        ),
        x=0.02,
        xanchor="left",
    ),
)

# Autoplay settings
fig_map_median.update_layout(
    updatemenus=[
        {
            "type": "buttons",
            "showactive": False,
            "x": 0.05,
            "y": 0,
            "xanchor": "left",
            "yanchor": "bottom",
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 900, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 350},
                        },
                    ],
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ]
)

# Display autoplay version inside notebook
map_div = pyo.plot(
    fig_map_median,
    include_plotlyjs="cdn",
    output_type="div",
    auto_play=True,
)

display(HTML(map_div))

In [ ]:
# Save autoplay HTML
pyo.plot(
    fig_map_median,
    filename=str(figure_output_dir / "autoplay_monthly_median_pm25_global_map.html"),
    include_plotlyjs="cdn",
    auto_play=True,
)

# Save map data
map_monthly.to_csv(
    web_data_output_dir / "map_monthly_median_pm25_2025.csv",
    index=False,
)

#### 2. City x month PM2.5 seasonality heatmap

In [24]:
# ============================================================
# 2. City x monthly PM2.5 seasonality heatmap
# ============================================================

heatmap_data = monthly.copy()

# City order by annual median PM2.5
city_order = (
    heatmap_data.groupby("city_display")[PM25_METRIC]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

heatmap_pivot = heatmap_data.pivot_table(
    index="city_display", columns="month_label", values=PM25_METRIC, aggfunc="median"
).reindex(index=city_order, columns=month_order)

coverage_pivot = heatmap_data.pivot_table(
    index="city_display",
    columns="month_label",
    values="pm25_month_coverage_pct",
    aggfunc="mean",
).reindex(index=city_order, columns=month_order)

valid_days_pivot = heatmap_data.pivot_table(
    index="city_display",
    columns="month_label",
    values="valid_pm25_days",
    aggfunc="mean",
).reindex(index=city_order, columns=month_order)

hover_text = []

for city in heatmap_pivot.index:
    row_text = []

    for month in heatmap_pivot.columns:
        pm_value = heatmap_pivot.loc[city, month]
        coverage_value = coverage_pivot.loc[city, month]
        valid_days = valid_days_pivot.loc[city, month]

        if pd.isna(pm_value):
            row_text.append(f"<b>{city}</b><br>{month}<br>No PM2.5 data")
        else:
            row_text.append(
                f"<b>{city}</b><br>"
                f"{month}<br>"
                f"Median PM2.5: {pm_value:.1f} µg/m³<br>"
                f"Valid PM2.5 days: {valid_days:.0f}<br>"
                f"Coverage: {coverage_value:.0f}%"
            )

    hover_text.append(row_text)

heatmap_color_max = np.nanpercentile(heatmap_pivot.values, 95)
heatmap_color_max = max(heatmap_color_max, 40)

fig_heatmap_median = go.Figure(
    data=go.Heatmap(
        z=heatmap_pivot.values,
        x=heatmap_pivot.columns,
        y=heatmap_pivot.index,
        colorscale="Inferno_r",
        colorbar=dict(title="Median<br>PM2.5<br>µg/m³"),
        text=hover_text,
        hoverinfo="text",
        zmin=0,
        zmax=heatmap_color_max,
    )
)

fig_heatmap_median.update_layout(
    title=dict(
        text=(
            "Seasonality Fingerprints: City × Month Median PM2.5 Heatmap"
            "<br><sup>Median is used for robust city comparison after removing invalid PM sentinel values.</sup>"
        ),
        x=0.02,
        xanchor="left",
    ),
    height=640,
    margin=dict(l=140, r=40, t=90, b=80),
    xaxis_title="Month",
    yaxis_title="City",
    template="plotly_white",
)

fig_heatmap_median.update_xaxes(side="bottom")
fig_heatmap_median.show()

fig_heatmap_median.write_html(
    figure_output_dir / "city_month_median_pm25_heatmap.html",
    include_plotlyjs="cdn",
)

heatmap_data.to_csv(
    web_data_output_dir / "city_month_median_pm25_heatmap_2025.csv",
    index=False,
)

## 3. Monthly PM2.5 rank transitions

Here we rank cities month by month using **monthly median PM2.5**.  
The goal is to show that city rankings are not fixed: some cities stay consistently high, while others move up or down with seasonal or episode-driven changes.

In [34]:
# ============================================================
# Ranking transition / bump chart using monthly median PM2.5
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ----------------------------
# Paths
# ----------------------------

project_root = Path(".")

combined_enriched_dir = project_root / "data" / "processed" / "enriched" / "combined"
city_selection_dir = project_root / "data" / "processed" / "combined" / "city_selection"

figure_output_dir = project_root / "outputs" / "figures"
web_data_output_dir = project_root / "web" / "data"

figure_output_dir.mkdir(parents=True, exist_ok=True)
web_data_output_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Load data
# ----------------------------

monthly_path = combined_enriched_dir / "combined_city_monthly_enriched_2025.csv"
selected_main_path = city_selection_dir / "selected_cities_pm25_main.csv"

monthly = pd.read_csv(monthly_path)
selected_main = pd.read_csv(selected_main_path)


# ----------------------------
# Helpers
# ----------------------------


def clean_city_key(value):
    return str(value).strip().lower().replace(" ", "_").replace("-", "_")


city_display_lookup = {
    "dubai": "Dubai",
    "riyadh": "Riyadh",
    "delhi": "Delhi",
    "lahore": "Lahore",
    "dhaka": "Dhaka",
    "bangkok": "Bangkok",
    "jakarta": "Jakarta",
    "singapore": "Singapore",
    "seoul": "Seoul",
    "tokyo": "Tokyo",
    "beijing": "Beijing",
    "london": "London",
    "los_angeles": "Los Angeles",
    "new_york": "New York",
    "mexico_city": "Mexico City",
}


city_to_country = {
    "dubai": "United Arab Emirates",
    "riyadh": "Saudi Arabia",
    "delhi": "India",
    "lahore": "Pakistan",
    "dhaka": "Bangladesh",
    "bangkok": "Thailand",
    "jakarta": "Indonesia",
    "singapore": "Singapore",
    "seoul": "South Korea",
    "tokyo": "Japan",
    "beijing": "China",
    "london": "United Kingdom",
    "los_angeles": "United States",
    "new_york": "United States",
    "mexico_city": "Mexico",
}


city_to_group = {
    "dubai": "Gulf / desert urbanization",
    "riyadh": "Gulf / desert urbanization",
    "delhi": "High-pollution / fast-growing Asia",
    "lahore": "High-pollution / fast-growing Asia",
    "dhaka": "High-pollution / fast-growing Asia",
    "bangkok": "High-pollution / fast-growing Asia",
    "jakarta": "High-pollution / fast-growing Asia",
    "singapore": "Dense but policy-relevant Asian comparison",
    "seoul": "Dense but policy-relevant Asian comparison",
    "tokyo": "Dense but policy-relevant Asian comparison",
    "beijing": "Dense but policy-relevant Asian comparison",
    "london": "Policy / developed-city comparison",
    "los_angeles": "Policy / developed-city comparison",
    "new_york": "Policy / developed-city comparison",
    "mexico_city": "Optional contrast",
}


def fill_city_metadata(df):
    df = df.copy()

    df["city_key"] = df["city"].apply(clean_city_key)
    df["city_display"] = df["city_key"].map(city_display_lookup).fillna(df["city"])

    if "country" not in df.columns:
        df["country"] = pd.NA

    if "city_group" not in df.columns:
        df["city_group"] = pd.NA

    missing_country = df["country"].isna() | df["country"].astype(
        str
    ).str.lower().str.strip().isin(["", "nan", "none", "null"])

    missing_group = df["city_group"].isna() | df["city_group"].astype(
        str
    ).str.lower().str.strip().isin(["", "nan", "none", "null"])

    df.loc[missing_country, "country"] = df.loc[missing_country, "city_key"].map(
        city_to_country
    )
    df.loc[missing_group, "city_group"] = df.loc[missing_group, "city_key"].map(
        city_to_group
    )

    return df


monthly = fill_city_metadata(monthly)
selected_main = fill_city_metadata(selected_main)

selected_city_keys = selected_main["city_key"].unique()
monthly = monthly[monthly["city_key"].isin(selected_city_keys)].copy()


# ----------------------------
# Prepare month/date fields
# ----------------------------

monthly["month_date"] = pd.to_datetime(
    monthly["local_year"].astype(int).astype(str)
    + "-"
    + monthly["local_month"].astype(int).astype(str).str.zfill(2)
    + "-01"
)

monthly["month_label"] = monthly["month_date"].dt.strftime("%b")

monthly = monthly.sort_values(["month_date", "city_display"]).copy()

PM25_RANK_METRIC = "pm25_monthly_median"

monthly[PM25_RANK_METRIC] = pd.to_numeric(monthly[PM25_RANK_METRIC], errors="coerce")

monthly.loc[monthly[PM25_RANK_METRIC] < 0, PM25_RANK_METRIC] = np.nan


# ----------------------------
# Rank cities within each month
# Rank 1 = highest median PM2.5
# ----------------------------

rank_data = monthly[monthly[PM25_RANK_METRIC].notna()].copy()

rank_data["pm25_rank"] = (
    rank_data.groupby("month_date")[PM25_RANK_METRIC]
    .rank(method="dense", ascending=False)
    .astype(int)
)

rank_data["rank_label"] = "#" + rank_data["pm25_rank"].astype(str)

rank_data["hover_text"] = (
    "<b>"
    + rank_data["city_display"]
    + "</b><br>"
    + "Month: "
    + rank_data["month_label"]
    + "<br>"
    + "Rank: #"
    + rank_data["pm25_rank"].astype(str)
    + "<br>"
    + "Median PM2.5: "
    + rank_data[PM25_RANK_METRIC].round(1).astype(str)
    + " µg/m³<br>"
    + "Valid PM2.5 days: "
    + rank_data["valid_pm25_days"].fillna(0).astype(int).astype(str)
    + "<br>"
    + "Coverage: "
    + rank_data["pm25_month_coverage_pct"].round(0).astype(str)
    + "%<br>"
    + "Country: "
    + rank_data["country"].astype(str)
)

# City order by average rank
city_order_by_avg_rank = (
    rank_data.groupby("city_display")["pm25_rank"].mean().sort_values().index.tolist()
)

rank_data["city_display"] = pd.Categorical(
    rank_data["city_display"], categories=city_order_by_avg_rank, ordered=True
)

rank_data = rank_data.sort_values(["city_display", "month_date"]).copy()


# ----------------------------
# Create bump chart
# ----------------------------

fig_bump = go.Figure()

# Highlight these cities with thicker lines
highlight_cities = [
    "Dubai",
    "Delhi",
    "Lahore",
    "Dhaka",
    "Seoul",
    "Bangkok",
]

for city in city_order_by_avg_rank:
    city_df = rank_data[rank_data["city_display"] == city].sort_values("month_date")

    is_highlight = city in highlight_cities

    fig_bump.add_trace(
        go.Scatter(
            x=city_df["month_label"],
            y=city_df["pm25_rank"],
            mode="lines+markers+text" if is_highlight else "lines+markers",
            name=city,
            text=city_df["rank_label"] if is_highlight else None,
            textposition="top center",
            line=dict(
                width=4 if is_highlight else 1.7,
                shape="spline",
            ),
            marker=dict(
                size=10 if is_highlight else 7,
                line=dict(width=1, color="white"),
            ),
            opacity=1.0 if is_highlight else 0.45,
            hovertext=city_df["hover_text"],
            hoverinfo="text",
        )
    )

max_rank = int(rank_data["pm25_rank"].max())

fig_bump.update_yaxes(
    autorange="reversed",
    tickmode="linear",
    tick0=1,
    dtick=1,
    range=[max_rank + 0.5, 0.5],
    title="Monthly rank<br><sup>1 = highest median PM2.5</sup>",
)

fig_bump.update_xaxes(
    title="Month in 2025",
    categoryorder="array",
    categoryarray=rank_data.sort_values("month_date")["month_label"]
    .drop_duplicates()
    .tolist(),
)

fig_bump.update_layout(
    title=dict(
        text=(
            "Rankings Are Not Enough: Monthly PM2.5 Rank Transitions"
            "<br><sup>Ranks use monthly median PM2.5 after invalid PM sentinel values were removed.</sup>"
        ),
        x=0.02,
        xanchor="left",
    ),
    height=760,
    template="plotly_white",
    margin=dict(l=80, r=180, t=90, b=70),
    legend=dict(
        title="City",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
    ),
)

fig_bump.show()


# ----------------------------
# Save outputs
# ----------------------------

fig_bump.write_html(
    figure_output_dir / "monthly_pm25_median_bump_chart.html",
    include_plotlyjs="cdn",
)

rank_data.to_csv(
    web_data_output_dir / "monthly_pm25_median_rank_transitions_2025.csv",
    index=False,
)

# print("Saved:")
# print(figure_output_dir / "monthly_pm25_median_bump_chart.html")
# print(web_data_output_dir / "monthly_pm25_median_rank_transitions_2025.csv")

The bump chart shows that PM2.5 rankings are not fixed across the year. Delhi, Dhaka, Lahore, Jakarta, and Dubai frequently occupy the highest ranks, but their positions shift month by month, suggesting that seasonal patterns and short pollution episodes matter more than a single annual ranking. Bangkok is especially dynamic, dropping from the top group early in the year to much lower ranks mid-year, then rising again in December.

## 4. WHO exceedance burden: daily PM2.5 dot-matrix

Here each dot is one city-day in 2025.  
We use **daily mean PM2.5** for WHO exceedance because the WHO guideline is defined as a 24-hour mean, while low-coverage days are shown separately.

In [37]:
# WHO PM2.5 daily exceedance dot-matrix

import sys
from pathlib import Path

project_root = Path(".")
analysis_scripts_dir = project_root / "scripts" / "analysis"

if str(analysis_scripts_dir) not in sys.path:
    sys.path.append(str(analysis_scripts_dir))

from exceedance_visuals import make_and_save_pm25_exceedance_dot_matrix

In [39]:
daily_path = "data/processed/enriched/combined/combined_city_daily_enriched_2025.csv"
selected_main_path = (
    "data/processed/combined/city_selection/selected_cities_pm25_main.csv"
)

fig_exceedance, daily_exceedance_matrix, exceedance_summary = (
    make_and_save_pm25_exceedance_dot_matrix(
        daily_path=daily_path,
        selected_main_path=selected_main_path,
        figure_output_dir="outputs/figures",
        web_data_output_dir="web/data",
        who_threshold=15,
        min_valid_hours=10,
        analysis_year=2025,
    )
)

fig_exceedance.show()

The dot-matrix makes the exceedance burden visible day by day. Delhi and Lahore exceed the WHO daily PM2.5 guideline almost continuously, while Jakarta and Dubai also show frequent exceedance days across much of the year. In contrast, New York, London, Tokyo, and Los Angeles have mostly below-guideline days, showing that annual averages alone miss the lived frequency of high-pollution days.

In [26]:
from episode_classification_pipeline import process_dubai_motor_city_episode_test

ModuleNotFoundError: No module named 'episode_classification_pipeline'

In [ ]:
episode_result = process_dubai_motor_city_episode_test(
    city_daily_enriched_input="data/processed/test_dubai_motor_city/city_daily_enriched.csv",
    output_dir="data/processed/test_dubai_motor_city",
    output_format="csv",
)

In [ ]:
episode_result["tables"].keys()

In [ ]:
episode_result["tables"]["city_daily_episodes"].head()